In [1]:
!pip install -q transformers accelerate openpyxl scikit-learn scipy matplotlib pandas
!pip install -q huggingface_hub
!pip install -q kernels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 7.1 MB/s eta 0:00:00


In [7]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

DRIVE_BASE = "/content/drive/MyDrive/Thesis"

Mounted at /content/drive


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import json
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist, mahalanobis
from scipy.special import rel_entr
from itertools import combinations
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME       = "openai/gpt-oss-20b"
EXCEL_PATH       = os.path.join(DRIVE_BASE, "Sentences.xlsx")
EXCEL_PATH_CS    = os.path.join(DRIVE_BASE, "comp_sci_sentences.xlsx")
EXCEL_PATH_HATE  = os.path.join(DRIVE_BASE, "Hate vs Not-Hate.xlsx")
RESULTS_DIR      = os.path.join(DRIVE_BASE, "results", "results_pca_gpt_oss_20b_sentences_prompted")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

COLORS = {
    "Shakespeare":           "#E06C75",
    "Computer Science":      "#61AFEF",
    "Information Security":  "#98C379",
    "Theory of Computation": "#E5C07B",
    "Hate Speech":           "#C678DD",
    "No-Hate Speech":        "#56B6C2",
}
CATEGORIES = list(COLORS.keys())

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [9]:
# ── Classification prompts (pair-specific) ───────────────────
# Each category gets the prompt appropriate for its pair.
# The model encodes the sentence in the context of distinguishing
# the two classes in that pair — we then pool only the sentence
# tokens (not the prompt tokens) to get the embedding.
PROMPTS = {
    "Shakespeare":           "Classify this text as Shakespeare or Computer Science:",
    "Computer Science":      "Classify this text as Shakespeare or Computer Science:",
    "Information Security":  "Classify this text as Information Security or Theory of Computation:",
    "Theory of Computation": "Classify this text as Information Security or Theory of Computation:",
    "Hate Speech":           "Classify this text as hate speech or not hate speech:",
    "No-Hate Speech":        "Classify this text as hate speech or not hate speech:",
}

print("\nPrompts:")
for cat, prompt in PROMPTS.items():
    print(f"  [{cat}]  {prompt}")


Prompts:
  [Shakespeare]  Classify this text as Shakespeare or Computer Science:
  [Computer Science]  Classify this text as Shakespeare or Computer Science:
  [Information Security]  Classify this text as Information Security or Theory of Computation:
  [Theory of Computation]  Classify this text as Information Security or Theory of Computation:
  [Hate Speech]  Classify this text as hate speech or not hate speech:
  [No-Hate Speech]  Classify this text as hate speech or not hate speech:


In [10]:
def load_sentences(excel_path, excel_path_cs):
    sentences = {}
    df1 = pd.read_excel(excel_path)
    for col in ["Shakespeare", "Computer Science"]:
        raw = df1[col].dropna().astype(str).tolist()
        sentences[col] = [s.strip() for s in raw if s.strip()]
    df2 = pd.read_excel(excel_path_cs)
    for col in ["Information Security", "Theory of Computation"]:
        raw = df2[col].dropna().astype(str).tolist()
        sentences[col] = [s.strip() for s in raw if s.strip()]
    return sentences


print("Loading sentences ...")
sentences = load_sentences(EXCEL_PATH, EXCEL_PATH_CS)
for cat, sents in sentences.items():
    print(f"  {cat}: {len(sents)} sentences")

Loading sentences ...
  Shakespeare: 100 sentences
  Computer Science: 100 sentences
  Information Security: 100 sentences
  Theory of Computation: 100 sentences


In [11]:
def load_model(model_name):
    print(f"Loading model: {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        output_hidden_states=True,
    )
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    hidden_size = model.config.hidden_size
    print(f"  Model loaded  (hidden dim = {hidden_size})")
    if hasattr(model, "hf_device_map"):
        print(f"  Device map: {model.hf_device_map}")
    return tokenizer, model


tokenizer, model = load_model(MODEL_NAME)

Loading model: openai/gpt-oss-20b ...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

...

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

  Model loaded  (hidden dim = 2880)


In [12]:
# CELL 7 — Extract prompted embeddings (sentence-token pooling)
# ============================================================
# Key difference from the unprompted script:
#   1. Each sentence is prefixed with its pair-specific classification prompt.
#   2. We tokenize the prompt alone to measure its token length.
#   3. Mean pooling is applied ONLY over the sentence tokens (not the prompt),
#      so the embedding reflects the sentence in the context of the task
#      without being contaminated by the prompt text itself.

def extract_prompted_embeddings(sentences_dict, tokenizer, model, prompts):
    all_embeddings, all_labels = [], []

    with torch.no_grad():
        for category, sents in sentences_dict.items():
            prompt     = prompts[category]
            prompt_ids = tokenizer(
                prompt,
                return_tensors="pt",
                add_special_tokens=False,
            )["input_ids"]
            n_prompt = prompt_ids.shape[1]   # number of prompt tokens

            print(f"  Encoding {len(sents)} '{category}' sentences  "
                  f"(prompt: {n_prompt} tokens) ...")

            for idx, sent in enumerate(sents):
                full_text = prompt + " " + sent

                inputs = tokenizer(
                    full_text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=192,       # longer than unprompted to fit prompt + sentence
                    padding="max_length",
                )

                first_device = next(model.parameters()).device
                inputs = {k: v.to(first_device) for k, v in inputs.items()}

                outputs     = model(**inputs)
                last_hidden = outputs.hidden_states[-1]   # (1, seq_len, hidden_dim)

                # Build sentence-only mask:
                #   attention_mask = 1 for real tokens (prompt + sentence), 0 for padding
                #   sentence_mask  = 1 only for sentence tokens (after prompt), 0 otherwise
                attn = inputs["attention_mask"].squeeze(0)          # (seq_len,)
                sentence_mask = attn.clone()
                sentence_mask[:n_prompt] = 0                        # zero out prompt positions

                # If the sentence was truncated and no sentence tokens remain, fall
                # back to pooling over all real tokens (edge case).
                if sentence_mask.sum() == 0:
                    sentence_mask = attn

                sent_mask_3d = sentence_mask.unsqueeze(0).unsqueeze(-1).to(
                    device=last_hidden.device, dtype=last_hidden.dtype
                )
                masked = last_hidden * sent_mask_3d
                pooled = masked.sum(dim=1) / sent_mask_3d.sum(dim=1)   # (1, hidden_dim)

                vec = pooled.squeeze(0).float().cpu().numpy()
                all_embeddings.append(vec)
                all_labels.append(category)

                if (idx + 1) % 10 == 0:
                    print(f"    [{category}] {idx+1}/{len(sents)} done")

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    embeddings = np.array(all_embeddings)
    labels     = np.array(all_labels)
    print(f"  Total embeddings: {embeddings.shape}")
    return embeddings, labels


print("Extracting prompted embeddings ...")
os.makedirs(RESULTS_DIR, exist_ok=True)
embeddings, labels = extract_prompted_embeddings(sentences, tokenizer, model, PROMPTS)

np.save(os.path.join(RESULTS_DIR, "embeddings.npy"), embeddings)
np.save(os.path.join(RESULTS_DIR, "labels.npy"), labels)
print(f"  Raw embeddings saved  ({embeddings.shape})")

# Free GPU memory
del model
del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("  Model unloaded, GPU memory freed.")

Extracting prompted embeddings ...
  Encoding 100 'Shakespeare' sentences  (prompt: 10 tokens) ...
    [Shakespeare] 10/100 done
    [Shakespeare] 20/100 done
    [Shakespeare] 30/100 done
    [Shakespeare] 40/100 done
    [Shakespeare] 50/100 done
    [Shakespeare] 60/100 done
    [Shakespeare] 70/100 done
    [Shakespeare] 80/100 done
    [Shakespeare] 90/100 done
    [Shakespeare] 100/100 done
  Encoding 100 'Computer Science' sentences  (prompt: 10 tokens) ...
    [Computer Science] 10/100 done
    [Computer Science] 20/100 done
    [Computer Science] 30/100 done
    [Computer Science] 40/100 done
    [Computer Science] 50/100 done
    [Computer Science] 60/100 done
    [Computer Science] 70/100 done
    [Computer Science] 80/100 done
    [Computer Science] 90/100 done
    [Computer Science] 100/100 done
  Encoding 100 'Information Security' sentences  (prompt: 13 tokens) ...
    [Information Security] 10/100 done
    [Information Security] 20/100 done
    [Information Security] 30

In [13]:
# ============================================================
# CELL 7b — Extract hate speech prompted embeddings (A100 only)
# ============================================================
# Run this block on A100 to extract hate speech embeddings with prompts.
# After saving, switch to CPU and use Cell 8 to reload everything.

tokenizer, model = load_model(MODEL_NAME)

df_hate = pd.read_excel(EXCEL_PATH_HATE)
hate_sentences = {}
for col in ["Hate Speech", "No-Hate Speech"]:
    raw = df_hate[col].dropna().astype(str).tolist()
    hate_sentences[col] = [s.strip() for s in raw if s.strip()]
for cat, sents in hate_sentences.items():
    print(f"  {cat}: {len(sents)} sentences")

hate_embeddings, hate_labels = extract_prompted_embeddings(hate_sentences, tokenizer, model, PROMPTS)
np.save(os.path.join(RESULTS_DIR, "hate_embeddings.npy"), hate_embeddings)
np.save(os.path.join(RESULTS_DIR, "hate_labels.npy"), hate_labels)
print(f"  Hate embeddings saved  ({hate_embeddings.shape})")

del model; del tokenizer
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("  Model unloaded.")

Loading model: openai/gpt-oss-20b ...


Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

  Model loaded  (hidden dim = 2880)
  Hate Speech: 100 sentences
  No-Hate Speech: 100 sentences
  Encoding 100 'Hate Speech' sentences  (prompt: 12 tokens) ...
    [Hate Speech] 10/100 done
    [Hate Speech] 20/100 done
    [Hate Speech] 30/100 done
    [Hate Speech] 40/100 done
    [Hate Speech] 50/100 done
    [Hate Speech] 60/100 done
    [Hate Speech] 70/100 done
    [Hate Speech] 80/100 done
    [Hate Speech] 90/100 done
    [Hate Speech] 100/100 done
  Encoding 100 'No-Hate Speech' sentences  (prompt: 12 tokens) ...
    [No-Hate Speech] 10/100 done
    [No-Hate Speech] 20/100 done
    [No-Hate Speech] 30/100 done
    [No-Hate Speech] 40/100 done
    [No-Hate Speech] 50/100 done
    [No-Hate Speech] 60/100 done
    [No-Hate Speech] 70/100 done
    [No-Hate Speech] 80/100 done
    [No-Hate Speech] 90/100 done
    [No-Hate Speech] 100/100 done
  Total embeddings: (200, 2880)
  Hate embeddings saved  ((200, 2880))
  Model unloaded.


In [14]:
import os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from scipy.spatial.distance import cdist, mahalanobis
from scipy.special import rel_entr
from itertools import combinations

if "DRIVE_BASE" not in globals():
    DRIVE_BASE = "/content/drive/MyDrive/thesis"
if "MODEL_NAME" not in globals():
    MODEL_NAME = "openai/gpt-oss-20b"
if "EXCEL_PATH" not in globals():
    EXCEL_PATH = os.path.join(DRIVE_BASE, "Sentences.xlsx")
if "EXCEL_PATH_CS" not in globals():
    EXCEL_PATH_CS = os.path.join(DRIVE_BASE, "comp_sci_sentences.xlsx")
if "EXCEL_PATH_HATE" not in globals():
    EXCEL_PATH_HATE = os.path.join(DRIVE_BASE, "Hate vs Not-Hate.xlsx")
if "RESULTS_DIR" not in globals():
    RESULTS_DIR = os.path.join(DRIVE_BASE, "results", "results_pca_gpt_oss_20b_sentences_prompted")
if "COLORS" not in globals():
    COLORS = {
        "Shakespeare":           "#E06C75",
        "Computer Science":      "#61AFEF",
        "Information Security":  "#98C379",
        "Theory of Computation": "#E5C07B",
        "Hate Speech":           "#C678DD",
        "No-Hate Speech":        "#56B6C2",
    }
if "CATEGORIES" not in globals():
    CATEGORIES = list(COLORS.keys())

os.makedirs(RESULTS_DIR, exist_ok=True)

# Uncomment to reload saved embeddings:
embeddings      = np.load(os.path.join(RESULTS_DIR, "embeddings.npy"))
labels          = np.load(os.path.join(RESULTS_DIR, "labels.npy"), allow_pickle=True)
hate_embeddings = np.load(os.path.join(RESULTS_DIR, "hate_embeddings.npy"))
hate_labels     = np.load(os.path.join(RESULTS_DIR, "hate_labels.npy"), allow_pickle=True)
print(f"Loaded embeddings: {embeddings.shape}, hate: {hate_embeddings.shape}")

Loaded embeddings: (400, 2880), hate: (200, 2880)


In [15]:
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances

all_embeddings = np.vstack([embeddings, hate_embeddings])
all_labels     = np.concatenate([labels, hate_labels])

ALL_CATS = ["Shakespeare", "Computer Science", "Information Security",
            "Theory of Computation", "Hate Speech", "No-Hate Speech"]

print(f"Combined embeddings: {all_embeddings.shape}")
for cat in ALL_CATS:
    print(f"  {cat}: {(all_labels == cat).sum()}")

def reload_sentences_map():
    sm = {}
    df1 = pd.read_excel(EXCEL_PATH)
    for col in ["Shakespeare", "Computer Science"]:
        raw = df1[col].dropna().astype(str).tolist()
        sm[col] = [s.strip() for s in raw if s.strip()]
    df2 = pd.read_excel(EXCEL_PATH_CS)
    for col in ["Information Security", "Theory of Computation"]:
        raw = df2[col].dropna().astype(str).tolist()
        sm[col] = [s.strip() for s in raw if s.strip()]
    df3 = pd.read_excel(EXCEL_PATH_HATE)
    for col in ["Hate Speech", "No-Hate Speech"]:
        raw = df3[col].dropna().astype(str).tolist()
        sm[col] = [s.strip() for s in raw if s.strip()]
    return sm

sentences_map = reload_sentences_map()
cat_counters  = {cat: 0 for cat in sentences_map}
all_sentences = []
for cat in all_labels:
    all_sentences.append(sentences_map[cat][cat_counters[cat]])
    cat_counters[cat] += 1
all_sentences = np.array(all_sentences)
print(f"Sentences aligned: {len(all_sentences)}")

# ── Subfolders ───────────────────────────────────────────────
DIR_DIST = os.path.join(RESULTS_DIR, "distance_metrics");  os.makedirs(DIR_DIST, exist_ok=True)
DIR_KL   = os.path.join(RESULTS_DIR, "kl_divergence");     os.makedirs(DIR_KL,   exist_ok=True)
DIR_SIL  = os.path.join(RESULTS_DIR, "silhouette");        os.makedirs(DIR_SIL,  exist_ok=True)
DIR_PCA  = os.path.join(RESULTS_DIR, "pca_plots");         os.makedirs(DIR_PCA,  exist_ok=True)
DIR_OVER = os.path.join(RESULTS_DIR, "overlap");           os.makedirs(DIR_OVER, exist_ok=True)
print("Subfolders ready.")

Combined embeddings: (600, 2880)
  Shakespeare: 100
  Computer Science: 100
  Information Security: 100
  Theory of Computation: 100
  Hate Speech: 100
  No-Hate Speech: 100
Sentences aligned: 600
Subfolders ready.


In [16]:
def compute_distances(emb, labs, tag, results_dir, n_pca=50):
    class_names = [c for c in ALL_CATS if c in set(labs)]
    class_data  = {c: emb[labs == c] for c in class_names}
    class_means = {c: d.mean(axis=0) for c, d in class_data.items()}

    n_components  = min(n_pca, emb.shape[0] - 1, emb.shape[1])
    pca50         = PCA(n_components=n_components)
    emb_r         = pca50.fit_transform(emb)
    var_explained = pca50.explained_variance_ratio_.sum()
    print(f"  PCA-{n_components} variance explained: {var_explained:.1%}  "
          f"(remaining {1-var_explained:.1%} discarded)")

    class_data_r  = {c: emb_r[labs == c] for c in class_names}
    class_means_r = {c: d.mean(axis=0) for c, d in class_data_r.items()}

    def _euc_cos(cdata, cmeans):
        intra_e, inter_e, inter_c = {}, {}, {}
        for c in class_names:
            d    = cdata[c]
            n    = d.shape[0]
            dmat = cdist(d, d, "euclidean")
            triu = np.triu_indices(n, k=1)
            intra_e[c] = dmat[triu].mean() if len(triu[0]) > 0 else 0.0
        for c1, c2 in combinations(class_names, 2):
            inter_e[(c1,c2)] = cdist(cdata[c1], cdata[c2], "euclidean").mean()
            m1n = cmeans[c1] / (np.linalg.norm(cmeans[c1]) + 1e-12)
            m2n = cmeans[c2] / (np.linalg.norm(cmeans[c2]) + 1e-12)
            inter_c[(c1,c2)] = 1.0 - float(np.dot(m1n, m2n))
        return intra_e, inter_e, inter_c

    intra_euc,   inter_euc,   inter_cos   = _euc_cos(class_data,   class_means)
    intra_euc_r, inter_euc_r, inter_cos_r = _euc_cos(class_data_r, class_means_r)

    pooled_cov = np.cov(emb_r, rowvar=False) + np.eye(n_components) * 1e-6
    try:
        cov_inv = np.linalg.inv(pooled_cov)
    except np.linalg.LinAlgError:
        cov_inv = np.linalg.pinv(pooled_cov)

    intra_mah, inter_mah = {}, {}
    for c in class_names:
        d = class_data_r[c]
        intra_mah[c] = np.mean([mahalanobis(d[i], class_means_r[c], cov_inv)
                                 for i in range(len(d))])
    for c1, c2 in combinations(class_names, 2):
        inter_mah[(c1,c2)] = mahalanobis(class_means_r[c1], class_means_r[c2], cov_inv)

    report = ["=" * 85,
              f"DISTANCE ANALYSIS  —  {tag}  [PROMPTED]",
              f"Model: {MODEL_NAME}",
              f"PCA-{n_components} variance explained: {var_explained:.1%}",
              "=" * 85]

    for space_label, intra_e, inter_e, inter_c, show_mah in [
        (f"FULL {emb.shape[1]}-d SPACE",  intra_euc,   inter_euc,   inter_cos,   False),
        (f"PCA-{n_components} SPACE",     intra_euc_r, inter_euc_r, inter_cos_r, True),
    ]:
        report.append(f"\n  ── {space_label} ──")
        report.append(f"  INTRA-CLASS:")
        header = f"  {'Class':<25} {'Avg Euclidean':>15} {'Avg Cosine':>12}"
        if show_mah: header += f" {'Mahalanobis':>13}"
        header += f" {'N':>6}"
        report.append(header)
        report.append(f"  {'-'*(len(header)-2)}")
        for c in class_names:
            row = f"  {c:<25} {intra_e[c]:>15.4f} {'N/A (centroid metric)':>12}"
            if show_mah: row += f" {intra_mah[c]:>13.4f}"
            row += f" {class_data[c].shape[0]:>6}"
            report.append(row)

        report.append(f"\n  INTER-CLASS:")
        header2 = f"  {'Pair':<40} {'Avg Euclidean':>15} {'Cosine':>8}"
        if show_mah: header2 += f" {'Mahalanobis':>13}"
        report.append(header2)
        report.append(f"  {'-'*(len(header2)-2)}")
        for c1, c2 in combinations(class_names, 2):
            pair = f"{c1} vs {c2}"
            row  = f"  {pair:<40} {inter_e[(c1,c2)]:>15.4f} {inter_c[(c1,c2)]:>8.4f}"
            if show_mah: row += f" {inter_mah[(c1,c2)]:>13.4f}"
            report.append(row)

        report.append(f"\n  SEPARABILITY RATIO (inter / avg intra)  —  Euclidean")
        if show_mah: report[-1] += " & Mahalanobis"
        header3 = f"  {'Pair':<40} {'Euc ratio':>10}"
        if show_mah: header3 += f" {'Mah ratio':>10}"
        report.append(header3)
        report.append(f"  {'-'*(len(header3)-2)}")
        for c1, c2 in combinations(class_names, 2):
            avg_ie = np.mean([intra_e[c1], intra_e[c2]])
            re     = inter_e[(c1,c2)] / avg_ie if avg_ie > 0 else float("inf")
            pair   = f"{c1} vs {c2}"
            row    = f"  {pair:<40} {re:>10.4f}"
            if show_mah:
                avg_im = np.mean([intra_mah[c1], intra_mah[c2]])
                rm     = inter_mah[(c1,c2)] / avg_im if avg_im > 0 else float("inf")
                row   += f" {rm:>10.4f}"
            report.append(row)

    report.append(f"\n{'='*85}")
    txt = "\n".join(report)
    print(txt)

    fname_txt = f"distance_analysis_{tag.lower().replace(' ','_')}.txt"
    with open(os.path.join(results_dir, fname_txt), "w", encoding="utf-8") as f:
        f.write(txt)
    print(f"  Saved {fname_txt}")

    n_cls = len(class_names)
    short = [c.replace("Theory of Computation","ToC").replace("Information Security","InfoSec")
              .replace("Computer Science","CS").replace("No-Hate Speech","No-Hate")
              .replace("Hate Speech","Hate") for c in class_names]

    heatmap_specs = [
        (f"Euclidean (full {emb.shape[1]}-d)",  intra_euc,   inter_euc,   "full_euclidean"),
        (f"Cosine (full {emb.shape[1]}-d)",      {c:0. for c in class_names}, inter_cos,   "full_cosine"),
        (f"Euclidean (PCA-{n_components})",      intra_euc_r, inter_euc_r, "pca_euclidean"),
        (f"Cosine (PCA-{n_components})",         {c:0. for c in class_names}, inter_cos_r, "pca_cosine"),
        (f"Mahalanobis (PCA-{n_components})",    intra_mah,   inter_mah,   "pca_mahalanobis"),
    ]
    for metric_name, intra_d, inter_d, metric_key in heatmap_specs:
        mat = np.zeros((n_cls, n_cls))
        for i, ci in enumerate(class_names):
            mat[i, i] = intra_d[ci]
            for j, cj in enumerate(class_names):
                if i < j:
                    mat[i, j] = inter_d[(ci, cj)]
                    mat[j, i] = inter_d[(ci, cj)]
        fig, ax = plt.subplots(figsize=(9, 7))
        im = ax.imshow(mat, cmap="YlOrRd", interpolation="nearest")
        plt.colorbar(im, ax=ax, shrink=0.8)
        for i in range(n_cls):
            for j in range(n_cls):
                color = "white" if mat[i, j] > mat.max() * 0.65 else "black"
                tag2  = "(intra)" if i == j else "(inter)"
                ax.text(j, i, f"{mat[i,j]:.3f}\n{tag2}",
                        ha="center", va="center", fontsize=9, fontweight="bold", color=color)
        ax.set_xticks(range(n_cls)); ax.set_xticklabels(short, fontsize=10, rotation=30, ha="right")
        ax.set_yticks(range(n_cls)); ax.set_yticklabels(short, fontsize=10)
        ax.set_title(f"{metric_name} — {tag} [Prompted]\nGPT-OSS-20B", fontsize=12, fontweight="bold")
        plt.tight_layout()
        fname_png = f"dist_{metric_key}_{tag.lower().replace(' ','_')}.png"
        plt.savefig(os.path.join(results_dir, fname_png), dpi=150, bbox_inches="tight")
        print(f"  Saved {fname_png}")
        plt.show()
        plt.close()


print("\n─── Distance Analysis: all 6 categories ───────────────────")
compute_distances(all_embeddings, all_labels, "All Categories", DIR_DIST)

PAIRS = [
    ("Computer Science", "Shakespeare"),
    ("Theory of Computation", "Information Security"),
    ("Hate Speech", "No-Hate Speech"),
]
for c1, c2 in PAIRS:
    mask = np.isin(all_labels, [c1, c2])
    print(f"\n─── Distance Analysis: {c1} vs {c2} ───")
    compute_distances(all_embeddings[mask], all_labels[mask], f"{c1} vs {c2}", DIR_DIST)


─── Distance Analysis: all 6 categories ───────────────────
  PCA-50 variance explained: 71.8%  (remaining 28.2% discarded)
DISTANCE ANALYSIS  —  All Categories  [PROMPTED]
Model: openai/gpt-oss-20b
PCA-50 variance explained: 71.8%

  ── FULL 2880-d SPACE ──
  INTRA-CLASS:
  Class                       Avg Euclidean   Avg Cosine      N
  -------------------------------------------------------------
  Shakespeare                       92.0716 N/A (centroid metric)    100
  Computer Science                 106.5290 N/A (centroid metric)    100
  Information Security             104.9585 N/A (centroid metric)    100
  Theory of Computation            102.8995 N/A (centroid metric)    100
  Hate Speech                       63.5347 N/A (centroid metric)    100
  No-Hate Speech                    67.7334 N/A (centroid metric)    100

  INTER-CLASS:
  Pair                                       Avg Euclidean   Cosine
  -----------------------------------------------------------------
  Shake

In [17]:
def compute_kl_divergence(emb, labs, tag, results_dir):
    class_names = [c for c in ALL_CATS if c in set(labs)]
    n_bins, eps = 50, 1e-10
    class_data  = {c: emb[labs == c] for c in class_names}
    g_min, g_max = emb.min(axis=0), emb.max(axis=0)

    class_dists = {}
    for c in class_names:
        dists = []
        for d in range(emb.shape[1]):
            vals   = class_data[c][:, d]
            lo, hi = g_min[d], g_max[d]
            if hi - lo < 1e-12:
                hist = np.ones(n_bins) / n_bins
            else:
                hist, _ = np.histogram(vals, bins=n_bins, range=(lo, hi))
                hist = hist.astype(np.float64) + eps
                hist /= hist.sum()
            dists.append(hist)
        class_dists[c] = np.array(dists)

    n_cls     = len(class_names)
    kl_matrix = np.zeros((n_cls, n_cls))
    report    = ["=" * 70, f"KL DIVERGENCE — {tag}  [PROMPTED]",
                 f"Model: {MODEL_NAME}", "=" * 70]
    report.append(f"\n  {'P \\ Q':<25} " + " ".join(f"{c:>15}" for c in class_names))
    report.append(f"  {'-'*(25 + 16*n_cls)}")

    for i, ci in enumerate(class_names):
        row = []
        for j, cj in enumerate(class_names):
            if i == j:
                kl_matrix[i, j] = 0.0
                row.append(f"{'0.0000':>15}")
            else:
                kl = np.mean([np.sum(rel_entr(class_dists[ci][k], class_dists[cj][k]))
                              for k in range(emb.shape[1])])
                kl_matrix[i, j] = kl
                row.append(f"{kl:>15.4f}")
        report.append(f"  {ci:<25} " + " ".join(row))

    report.append(f"\n{'='*70}")
    txt = "\n".join(report)
    print(txt)

    fname_txt = f"kl_divergence_{tag.lower().replace(' ','_')}.txt"
    with open(os.path.join(results_dir, fname_txt), "w", encoding="utf-8") as f:
        f.write(txt)
    print(f"  Saved {fname_txt}")

    short = [c.replace("Theory of Computation","ToC").replace("Information Security","InfoSec")
              .replace("Computer Science","CS").replace("No-Hate Speech","No-Hate")
              .replace("Hate Speech","Hate") for c in class_names]

    fig, ax = plt.subplots(figsize=(max(7, n_cls*1.4), max(6, n_cls*1.2)))
    im = ax.imshow(kl_matrix, cmap="Blues", interpolation="nearest")
    plt.colorbar(im, ax=ax, shrink=0.8, label="KL Divergence (nats)")
    for i in range(n_cls):
        for j in range(n_cls):
            color = "white" if kl_matrix[i, j] > kl_matrix.max() * 0.65 else "black"
            ax.text(j, i, f"{kl_matrix[i,j]:.3f}",
                    ha="center", va="center", fontsize=10, fontweight="bold", color=color)
    ax.set_xticks(range(n_cls)); ax.set_xticklabels(short, fontsize=10, rotation=30, ha="right")
    ax.set_yticks(range(n_cls)); ax.set_yticklabels(short, fontsize=10)
    ax.set_xlabel("Q (reference)", fontsize=11)
    ax.set_ylabel("P (source)", fontsize=11)
    ax.set_title(f"KL Divergence — {tag} [Prompted]\nGPT-OSS-20B  |  KL(P‖Q) avg over neurons",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    fname_png = f"kl_divergence_{tag.lower().replace(' ','_')}.png"
    plt.savefig(os.path.join(results_dir, fname_png), dpi=150, bbox_inches="tight")
    print(f"  Saved {fname_png}")
    plt.show()
    plt.close()


print("\n─── KL Divergence: all 6 categories ───────────────────────")
compute_kl_divergence(all_embeddings, all_labels, "All Categories", DIR_KL)

for c1, c2 in PAIRS:
    mask = np.isin(all_labels, [c1, c2])
    print(f"\n─── KL Divergence: {c1} vs {c2} ───")
    compute_kl_divergence(all_embeddings[mask], all_labels[mask], f"{c1} vs {c2}", DIR_KL)


─── KL Divergence: all 6 categories ───────────────────────
KL DIVERGENCE — All Categories  [PROMPTED]
Model: openai/gpt-oss-20b

  P \ Q                         Shakespeare Computer Science Information Security Theory of Computation     Hate Speech  No-Hate Speech
  -------------------------------------------------------------------------------------------------------------------------
  Shakespeare                        0.0000          3.3881          4.1865          4.4076          7.7594          7.4299
  Computer Science                   4.5985          0.0000          3.2104          3.5558         10.0658          9.7648
  Information Security               5.4193          3.1593          0.0000          3.6632          9.8853          9.5807
  Theory of Computation              5.4409          3.3545          3.4267          0.0000         10.0806          9.8026
  Hate Speech                        3.2060          4.2823          4.0168          4.4834          0.0000      

In [18]:
def compute_silhouette(emb, labs, tag, results_dir, n_pca=50):
    pca_red = PCA(n_components=min(n_pca, emb.shape[0]-1, emb.shape[1]))
    emb_r   = pca_red.fit_transform(emb)

    score   = silhouette_score(emb_r, labs)
    samples = silhouette_samples(emb_r, labs)

    class_names = [c for c in ALL_CATS if c in set(labs)]
    report = [
        "=" * 60,
        f"SILHOUETTE SCORES — {tag}  [PROMPTED]",
        f"Model: {MODEL_NAME}  |  PCA reduced to {emb_r.shape[1]} dims",
        "=" * 60,
        f"\n  Overall silhouette score: {score:.4f}",
        f"  (range -1 to +1; higher = better separated)\n",
        f"  {'Class':<25} {'Mean silhouette':>16} {'N':>6}",
        f"  {'-'*49}",
    ]
    for c in class_names:
        idx = labs == c
        report.append(f"  {c:<25} {samples[idx].mean():>16.4f} {idx.sum():>6}")
    report.append(f"\n{'='*60}")
    txt = "\n".join(report)
    print(txt)

    fname = f"silhouette_{tag.lower().replace(' ','_')}.txt"
    with open(os.path.join(results_dir, fname), "w", encoding="utf-8") as f:
        f.write(txt)
    print(f"  Saved {fname}")

    class_scores = [samples[labs == c].mean() for c in class_names]
    short = [c.replace("Theory of Computation","ToC").replace("Information Security","InfoSec")
              .replace("Computer Science","CS").replace("No-Hate Speech","No-Hate")
              .replace("Hate Speech","Hate") for c in class_names]
    bar_colors = [COLORS[c] for c in class_names]

    fig, ax = plt.subplots(figsize=(max(7, len(class_names)*1.3), 5))
    bars = ax.bar(short, class_scores, color=bar_colors, edgecolor="white", linewidth=0.8)
    ax.axhline(score, color="black", linestyle="--", linewidth=1.2, label=f"Overall: {score:.3f}")
    ax.axhline(0, color="grey", linestyle=":", linewidth=0.8)
    for bar, val in zip(bars, class_scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_ylabel("Mean Silhouette Score", fontsize=11)
    ax.set_title(f"Silhouette Scores — {tag} [Prompted]\nGPT-OSS-20B (PCA-{emb_r.shape[1]})",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.set_ylim(min(class_scores) - 0.05, max(class_scores) + 0.08)
    plt.tight_layout()
    fname_png = f"silhouette_{tag.lower().replace(' ','_')}.png"
    plt.savefig(os.path.join(results_dir, fname_png), dpi=150, bbox_inches="tight")
    print(f"  Saved {fname_png}")
    plt.show()
    plt.close()
    return score


print("\n─── Silhouette: all 6 categories ───────────────────────────")
compute_silhouette(all_embeddings, all_labels, "All Categories", DIR_SIL)

for c1, c2 in PAIRS:
    mask = np.isin(all_labels, [c1, c2])
    print(f"\n─── Silhouette: {c1} vs {c2} ───")
    compute_silhouette(all_embeddings[mask], all_labels[mask], f"{c1} vs {c2}", DIR_SIL)


─── Silhouette: all 6 categories ───────────────────────────
SILHOUETTE SCORES — All Categories  [PROMPTED]
Model: openai/gpt-oss-20b  |  PCA reduced to 50 dims

  Overall silhouette score: 0.1362
  (range -1 to +1; higher = better separated)

  Class                      Mean silhouette      N
  -------------------------------------------------
  Shakespeare                         0.3696    100
  Computer Science                    0.1074    100
  Information Security                0.1555    100
  Theory of Computation               0.1738    100
  Hate Speech                         0.0444    100
  No-Hate Speech                     -0.0336    100

  Saved silhouette_all_categories.txt
  Saved silhouette_all_categories.png

─── Silhouette: Computer Science vs Shakespeare ───
SILHOUETTE SCORES — Computer Science vs Shakespeare  [PROMPTED]
Model: openai/gpt-oss-20b  |  PCA reduced to 50 dims

  Overall silhouette score: 0.2291
  (range -1 to +1; higher = better separated)

  Class  

In [19]:
def pca_plot(emb, labs, cats, title, fname_base, results_dir):
    mask    = np.isin(labs, cats)
    emb_sub = emb[mask]
    lab_sub = labs[mask]

    pca2 = PCA(n_components=2)
    X2   = pca2.fit_transform(emb_sub)
    evr2 = pca2.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(10, 8))
    for cat in cats:
        idx = lab_sub == cat
        ax.scatter(X2[idx, 0], X2[idx, 1], c=COLORS[cat], label=cat,
                   alpha=0.75, edgecolors="w", linewidths=0.5, s=60)
    ax.set_xlabel(f"PC 1  ({evr2[0]:.1%})", fontsize=12)
    ax.set_ylabel(f"PC 2  ({evr2[1]:.1%})", fontsize=12)
    ax.set_title(f"GPT-OSS-20B — 2D PCA [Prompted]\n{title}", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10, loc="best")
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    p2 = os.path.join(results_dir, f"{fname_base}_2d.png")
    plt.savefig(p2, dpi=150, bbox_inches="tight")
    print(f"  Saved {p2}")
    plt.show(); plt.close()

    pca3 = PCA(n_components=3)
    X3   = pca3.fit_transform(emb_sub)
    evr3 = pca3.explained_variance_ratio_

    fig = plt.figure(figsize=(12, 10))
    ax  = fig.add_subplot(111, projection="3d")
    for cat in cats:
        idx = lab_sub == cat
        ax.scatter(X3[idx,0], X3[idx,1], X3[idx,2], c=COLORS[cat], label=cat,
                   alpha=0.75, edgecolors="w", linewidths=0.5, s=60)
    ax.set_xlabel(f"PC 1  ({evr3[0]:.1%})", fontsize=10)
    ax.set_ylabel(f"PC 2  ({evr3[1]:.1%})", fontsize=10)
    ax.set_zlabel(f"PC 3  ({evr3[2]:.1%})", fontsize=10)
    ax.set_title(f"GPT-OSS-20B — 3D PCA [Prompted]\n{title}", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10, loc="best")
    plt.tight_layout()
    p3 = os.path.join(results_dir, f"{fname_base}_3d.png")
    plt.savefig(p3, dpi=150, bbox_inches="tight")
    print(f"  Saved {p3}")
    plt.show(); plt.close()


print("\n─── PCA: all 6 categories ──────────────────────────────────")
pca_plot(all_embeddings, all_labels, ALL_CATS,
         "All 6 Categories", "pca_all", DIR_PCA)

print("\n─── PCA: pairwise ──────────────────────────────────────────")
pair_meta = [
    (["Computer Science", "Shakespeare"],               "CS vs Shakespeare",          "pca_cs_shakespeare"),
    (["Theory of Computation", "Information Security"], "ToC vs Information Security", "pca_toc_infosec"),
    (["Hate Speech", "No-Hate Speech"],                 "Hate vs No-Hate Speech",     "pca_hate"),
]
for cats, title, fname in pair_meta:
    print(f"\n  {title}")
    pca_plot(all_embeddings, all_labels, cats, title, fname, DIR_PCA)


─── PCA: all 6 categories ──────────────────────────────────
  Saved /content/drive/MyDrive/Thesis/results/results_pca_gpt_oss_20b_sentences_prompted/pca_plots/pca_all_2d.png
  Saved /content/drive/MyDrive/Thesis/results/results_pca_gpt_oss_20b_sentences_prompted/pca_plots/pca_all_3d.png

─── PCA: pairwise ──────────────────────────────────────────

  CS vs Shakespeare
  Saved /content/drive/MyDrive/Thesis/results/results_pca_gpt_oss_20b_sentences_prompted/pca_plots/pca_cs_shakespeare_2d.png
  Saved /content/drive/MyDrive/Thesis/results/results_pca_gpt_oss_20b_sentences_prompted/pca_plots/pca_cs_shakespeare_3d.png

  ToC vs Information Security
  Saved /content/drive/MyDrive/Thesis/results/results_pca_gpt_oss_20b_sentences_prompted/pca_plots/pca_toc_infosec_2d.png
  Saved /content/drive/MyDrive/Thesis/results/results_pca_gpt_oss_20b_sentences_prompted/pca_plots/pca_toc_infosec_3d.png

  Hate vs No-Hate Speech
  Saved /content/drive/MyDrive/Thesis/results/results_pca_gpt_oss_20b_senten

In [20]:
def find_overlapping_sentences(emb, labs, sents, c1, c2, top_k=5, results_dir=None, n_pca=50):
    idx1 = np.where(labs == c1)[0]
    idx2 = np.where(labs == c2)[0]

    dmat_full = cdist(emb[idx1], emb[idx2], "euclidean")

    n_components  = min(n_pca, emb.shape[0] - 1, emb.shape[1])
    pca50         = PCA(n_components=n_components)
    emb_r         = pca50.fit_transform(emb)
    var_explained = pca50.explained_variance_ratio_.sum()
    dmat_pca      = cdist(emb_r[idx1], emb_r[idx2], "euclidean")

    report = [
        "=" * 80,
        f"MOST OVERLAPPING SENTENCE PAIRS  —  {c1}  vs  {c2}  [PROMPTED]",
        "=" * 80,
    ]
    for space_label, dmat in [
        (f"Full {emb.shape[1]}-d Euclidean", dmat_full),
        (f"PCA-{n_components} Euclidean  (variance explained: {var_explained:.1%})", dmat_pca),
    ]:
        report.append(f"\n  ── {space_label} ──")
        flat = dmat.flatten()
        top  = np.argsort(flat)[:top_k]
        for rank, flat_idx in enumerate(top, 1):
            i, j  = np.unravel_index(flat_idx, dmat.shape)
            dist  = dmat[i, j]
            s1    = sents[idx1[i]]
            s2    = sents[idx2[j]]
            report.append(f"\n  Rank {rank}  |  distance = {dist:.4f}")
            report.append(f"  [{c1}]  {s1}")
            report.append(f"  [{c2}]  {s2}")

    txt = "\n".join(report)
    print(txt)
    if results_dir:
        safe  = f"{c1}_{c2}".lower().replace(" ", "_")
        fname = os.path.join(results_dir, f"overlap_{safe}.txt")
        with open(fname, "w", encoding="utf-8") as f:
            f.write(txt)
        print(f"\n  Saved {fname}")


print("\n─── Overlapping sentences ──────────────────────────────────")
for c1, c2 in PAIRS:
    find_overlapping_sentences(all_embeddings, all_labels, all_sentences,
                               c1, c2, top_k=5, results_dir=DIR_OVER)


─── Overlapping sentences ──────────────────────────────────
MOST OVERLAPPING SENTENCE PAIRS  —  Computer Science  vs  Shakespeare  [PROMPTED]

  ── Full 2880-d Euclidean ──

  Rank 1  |  distance = 80.8800
  [Computer Science]  An algorithm is a finite sequence of well-defined instructions to solve a problem.
  [Shakespeare]  The worst is not so long as we can say, “This is the worst.”

  Rank 2  |  distance = 82.0669
  [Computer Science]  Overfitting occurs when a model learns noise instead of patterns.
  [Shakespeare]  The worst is not so long as we can say, “This is the worst.”

  Rank 3  |  distance = 82.6755
  [Computer Science]  Machine learning enables systems to learn from data.
  [Shakespeare]  Things done well and with a care, exempt themselves from fear.

  Rank 4  |  distance = 82.9641
  [Computer Science]  Processes are independent execution units in a system.
  [Shakespeare]  Things done well and with a care, exempt themselves from fear.

  Rank 5  |  distance = 82.9956

In [21]:
# ============================================================
# CELL 15 — Summary JSON
# ============================================================
summary = {
    "model":         MODEL_NAME,
    "embedding_dim": int(all_embeddings.shape[1]),
    "n_sentences":   {cat: int((all_labels == cat).sum()) for cat in ALL_CATS},
    "prompts":       PROMPTS,
}
with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*60}")
print(f"All results saved to: {RESULTS_DIR}")
print(f"{'='*60}")


All results saved to: /content/drive/MyDrive/Thesis/results/results_pca_gpt_oss_20b_sentences_prompted
